# bag0 v5: friction landscape on the push parts at grid 96 (forward-only sweep)
Runs the TaichiDough objective for 8 x 8 tool-friction x floor-friction sets on p03 and p05 (v5 grid box, grid 96). Needs a GPU runtime (Runtime > Change runtime type > GPU). Set `DRIVE_FILE_ID` to the id of `bag0_dynamics_v5_gpu_package.zip` in your Google Drive (share link .../d/<id>/view).

In [ ]:
DRIVE_FILE_ID = 'PASTE_FILE_ID_HERE'  # bag0_dynamics_v5_gpu_package.zip in Google Drive
BRANCH = 'bounded-shear-adhesion-v3'
PARTS = 'bag0_p03_E1.3_push bag0_p05_E1.5_push'
GRID = 96

In [ ]:
!nvidia-smi -L
!pip -q install taichi==1.7.4 open3d==0.20.0 scipy gdown
import taichi, open3d; print(taichi.__version__, open3d.__version__)

In [ ]:
%cd /content
!rm -rf TaichiDough && git clone -q --branch $BRANCH https://github.com/AntonioSkara132/TaichiDough.git
%cd /content/TaichiDough
!git log --oneline -1

In [ ]:
!gdown --id $DRIVE_FILE_ID -O /content/bag0_v5.zip
import zipfile
z = zipfile.ZipFile('/content/bag0_v5.zip')
z.extractall('/content/TaichiDough', [n for n in z.namelist() if n.startswith('experiments/')])
print('extracted')

In [ ]:
# same two local patches as the GPU host: cache identity version pins; SDF tolerance pinned to the grid-48 value
import pathlib
p = pathlib.Path('experiments/differentiable_mpm/sdf_cache.py'); s = p.read_text()
old = '    return hashlib.sha256(_json(description).encode()).hexdigest(), description'
new = '\n'.join([
    '    def _pin(v):',
    '        if isinstance(v, dict): return {k: ("2.2.6" if k == "numpy_version" else "0.17.0" if k == "open3d_version" else _pin(x)) for k, x in v.items()}',
    '        if isinstance(v, list): return [_pin(x) for x in v]',
    '        return v',
    '    description = _pin(description)',
    old])
assert old in s; p.write_text(s.replace(old, new, 1))
p = pathlib.Path('experiments/differentiable_mpm/data.py'); s = p.read_text()
old = '            settings = qualification_settings(config.tool_sdf_qualification, grid_spacing)'
new = '            grid_spacing = 1.0 / 48  # tolerance pinned to grid 48 (shipped 256^3 fields)\n' + old
assert old in s; p.write_text(s.replace(old, new, 1)); print('patched')

In [ ]:
# grid-96 box configs (box indices x2) for all parts, and the friction grid
import json
D = 'experiments/differentiable_mpm/data/bag0_dynamics_v5/'
d = json.load(open(D + 'dataset_bag0_all_von_mises_1000pa_coulomb_v5.json')); f = GRID // 48; eps = []
for e in d['episodes']:
    c = json.load(open(D + e['config'])); c['simulation']['grid'] = GRID
    c['simulation']['grid_box_lower'] = [v * f for v in c['simulation']['grid_box_lower']]; c['simulation']['grid_box_cells'] = [v * f for v in c['simulation']['grid_box_cells']]
    new = e['config'].replace('_coulomb_v5.json', '_coulomb_v5_g%d.json' % GRID); json.dump(c, open(D + new, 'w'), indent=2); eps.append(dict(e, config=new))
dd = dict(d); dd['name'] += '-grid%dbox' % GRID; dd['episodes'] = eps; json.dump(dd, open(D + 'dataset_v5_g%d.json' % GRID, 'w'), indent=2)
json.dump({'base': {}, 'axes': {'tool_friction_coefficient': [0.1, 0.2, 0.3, 0.45, 0.6, 0.8, 1.1, 1.5], 'floor_friction_coefficient': [0.1, 0.2, 0.3, 0.45, 0.6, 0.8, 1.1, 1.5]}}, open('/content/grid_friction.json', 'w'))
print('ok')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/bag0_colab

In [ ]:
%%writefile /content/sweep_bag0.py
"""Grid sweep of fixed parameter sets through the fits' own objective (same loss as calibrate_dataset), one process per episode list.
Each episode is prepared once and its rollout compiled once; then every parameter set is evaluated without gradients.

usage: sweep.py CODE_ROOT DATASET_JSON GRID_JSON OUT_JSONL EPISODE_ID [EPISODE_ID ...]
GRID_JSON: {"base": {...shared parameters...}, "axes": {"von_mises_yield_stress_pa": [...], "youngs_modulus": [...], "viscosity": [...]}}
Resumable: (episode, parameter-set) pairs already in OUT_JSONL are skipped. Rows: episode, membership, set, value, frames, diagnostics summary, seconds."""
import itertools, json, os, sys, time
root, dataset_path, grid_path, out_path = sys.argv[1:5]; episodes = [e for e in sys.argv[5:] if not e.startswith("--")]
shard = next((e for e in sys.argv[5:] if e.startswith("--shard=")), "--shard=0/1")[8:]; shard_i, shard_n = map(int, shard.split("/"))
precision = next((e[12:] for e in sys.argv[5:] if e.startswith("--precision=")), "f32")
stride = int(next((e[9:] for e in sys.argv[5:] if e.startswith("--stride=")), "1"))
os.chdir(root); sys.path.insert(0, root)
from experiments.differentiable_mpm.calibrate import make_rollout
from experiments.differentiable_mpm.data import prepare_experiment
from experiments.differentiable_mpm.dataset_config import load_dataset
from experiments.differentiable_mpm.reference_adapter import reference_policy
from experiments.differentiable_mpm.runtime import init_runtime
grid = json.load(open(grid_path)); names = list(grid["axes"])
sets = [dict(zip(names, combo)) for combo in itertools.product(*(grid["axes"][n] for n in names))]
sets = sets[shard_i::shard_n]
done = set()
if os.path.exists(out_path):
    for line in open(out_path):
        r = json.loads(line); done.add((r["episode"], json.dumps(r["set"], sort_keys=True)))
quiet = lambda *a, **k: None
with reference_policy("frozen"):
    ds = load_dataset(dataset_path, backend="cuda", precision=precision, p2g_mode="atomic")
    byid = {e.id: e for e in ds.episodes}
    first = byid[episodes[0]].config
    init_runtime("cuda", precision, cpu_threads=1, seed=first.seed)
    for eid in episodes:
        ep = byid[eid]; t0 = time.perf_counter()
        window = ep.scored_window if stride == 1 else type(ep.scored_window)(ep.scored_window.start_frame, ep.scored_window.end_frame, stride)
        prepared = prepare_experiment(ep.config, split=ep.membership, build_sdf=True, scored_window=window)
        _, rollout = make_rollout(prepared, progress=quiet)
        print(f"{eid}: shard {shard_i}/{shard_n}, {len(sets)} sets, prepared in {time.perf_counter() - t0:.0f} s, precision {prepared.simulation_config.precision}, {prepared.total_steps} steps", flush=True)
        for s in sets:
            key = json.dumps(s, sort_keys=True)
            if (eid, key) in done: continue
            shared = dict(ds.shared_initial); shared.update(grid.get("base", {})); shared.update(s); extra = {k: shared.pop(k) for k in list(shared) if k not in ep.shared_parameter_names}; params = ep.parameters_for(shared); params.update(extra); t1 = time.perf_counter()
            try:
                ev = rollout.value_and_gradient(params, compute_grad=False); value = float(ev.value); err = None
                diag = {k: ev.diagnostics.get(k) for k in ("branch_counts_total",) if k in ev.diagnostics}
            except Exception as exc:
                value, err, diag = None, f"{type(exc).__name__}: {exc}"[:300], {}
            row = {"episode": eid, "membership": ep.membership, "set": s, "value": value, "error": err, "seconds": round(time.perf_counter() - t1, 1), **diag}
            with open(out_path, "a") as f: f.write(json.dumps(row) + "\n")
            print(f"  {s} -> {value if value is None else round(value, 4)} ({row['seconds']} s){'  ' + err if err else ''}", flush=True)


In [ ]:
import os; os.environ['TAICHIDOUGH_SDF_CACHE'] = '/content/TaichiDough/experiments/differentiable_mpm/data/bag0_dynamics_v5_sdf_cache'
!cd /content/TaichiDough && python /content/sweep_bag0.py /content/TaichiDough experiments/differentiable_mpm/data/bag0_dynamics_v5/dataset_v5_g$GRID.json /content/grid_friction.json /content/drive/MyDrive/bag0_colab/friction_pushes_g$GRID.jsonl $PARTS --shard=0/1 --precision=f32 2>&1 | grep -v '^\[Taichi\]\|^\[W '

In [ ]:
import json, collections, numpy as np
rows = [json.loads(l) for l in open('/content/drive/MyDrive/bag0_colab/friction_pushes_g%d.jsonl' % GRID)]
by = collections.defaultdict(list)
for r in rows:
    if r['value'] is not None: by[r['episode']].append((r['value'], r['set']))
for ep, v in by.items():
    v.sort(); print(ep, 'best', v[0], 'worst', v[-1][0])
    for ax in ('tool_friction_coefficient', 'floor_friction_coefficient'):
        m = collections.defaultdict(list)
        for val, s in v: m[s[ax]].append(val)
        print('  marginal by', ax, {k: round(float(np.mean(x)), 4) for k, x in sorted(m.items())})